# KNN Content-Based Course Recommender

## 1. Introduction

This notebook builds a KNN-based recommender for the Online Course Recommender System project.

The goal is to recommend courses that are similar to a selected course by comparing numerical feature vectors. The notebook follows the processed dataset created in `02_preprocessing.ipynb` and complements the TF-IDF recommender in `03_tfidf_recommender.ipynb`.

## 2. Load Processed Dataset

The processed dataset is loaded from `data/processed/processed_udemy_courses.csv`.

This notebook expects the preprocessing step to have already created the selected recommendation columns, including `combined_features`.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler

In [2]:
DATA_PATHS = [
    Path("data/processed/processed_udemy_courses.csv"),
    Path("../data/processed/processed_udemy_courses.csv"),
]

DATA_PATH = next((path for path in DATA_PATHS if path.exists()), None)
if DATA_PATH is None:
    raise FileNotFoundError("Processed dataset not found. Run 02_preprocessing.ipynb first.")

df = pd.read_csv(DATA_PATH)

print(f"Loaded dataset from: {DATA_PATH}")
print(f"Dataset shape: {df.shape[0]} rows x {df.shape[1]} columns")

Loaded dataset from: ..\data\processed\processed_udemy_courses.csv
Dataset shape: 3672 rows x 9 columns


The dataset shape confirms how many courses and features are available after preprocessing. The next checks make sure the columns required for KNN are present before model building starts.

In [3]:
important_columns = [
    "course_id",
    "course_title",
    "subject",
    "level",
    "num_subscribers",
    "num_reviews",
    "price",
    "content_duration",
    "combined_features",
]

df[important_columns].head()

,course_id,course_title,subject,level,num_subscribers,num_reviews,price,content_duration,combined_features
0,1070968,ultimate investment banking course,business finance,all levels,2147,23,200,1.5,ultimate investment banking course business fi...
1,1113822,complete gst course certification grow your ca...,business finance,all levels,2792,923,75,39.0,complete gst course certification grow your ca...
2,1006314,financial modeling for business analysts and c...,business finance,intermediate level,2174,74,45,2.5,financial modeling for business analysts and c...
3,1210588,beginner to pro financial analysis in excel 2017,business finance,all levels,2451,11,95,3.0,beginner to pro financial analysis in excel 20...
4,1011058,how to maximize your profits trading options,business finance,intermediate level,1276,45,200,2.0,how to maximize your profits trading options b...


In [4]:
missing_columns = [column for column in important_columns if column not in df.columns]

if missing_columns:
    raise ValueError(f"Missing required columns: {missing_columns}")

print("All required columns are available for the KNN recommender.")
print(f"Missing combined_features values: {df['combined_features'].isna().sum()}")

All required columns are available for the KNN recommender.
Missing combined_features values: 0


## 3. Feature Preparation

KNN compares rows using distances between feature vectors. For this reason, each course needs to be represented with numerical values.

The planned feature set combines text information from `combined_features` with numeric course attributes such as subscribers, reviews, price, and duration.

In [5]:
text_feature = "combined_features"
numeric_features = [
    "num_subscribers",
    "num_reviews",
    "price",
    "content_duration",
]

df[["course_title", "subject", "level", text_feature] + numeric_features].sample(
    5, random_state=42
)

,course_title,subject,level,combined_features,num_subscribers,num_reviews,price,content_duration
2154,play guitar and understand music quick easy in...,musical instruments,all levels,play guitar and understand music quick easy in...,738,6,20,9.5
2740,wordpress blog create a wordpress website for ...,web development,beginner level,wordpress blog create a wordpress website for ...,2076,8,20,2.0
1632,the most popular techniques in photoshop,graphic design,intermediate level,the most popular techniques in photoshop graph...,1753,18,20,1.0
2003,guitar blues guitar for beginners,musical instruments,beginner level,guitar blues guitar for beginners musical inst...,2795,26,45,1.0
3248,modern react with redux,web development,all levels,modern react with redux web development all le...,50815,15117,180,26.5


The text feature gives the main content signal for the recommender. The numeric features add extra course context, such as course popularity, price, and duration.

`subject` and `level` are already included inside `combined_features`, so they do not need to be encoded separately in this first KNN version.

In [6]:
df[numeric_features].describe()

,num_subscribers,num_reviews,price,content_duration
count,3672.000000,3672.000000,3672.000000,3672.000000
mean,3190.586874,156.371460,66.102941,4.097603
std,9488.105448,936.178649,61.035920,6.057830
min,0.000000,0.000000,0.000000,0.000000
25%,111.750000,4.000000,20.000000,1.000000
50%,912.000000,18.000000,45.000000,2.000000
75%,2548.750000,67.000000,95.000000,4.500000
max,268923.000000,27445.000000,200.000000,78.500000


## 4. Vectorization / Feature Transformation

Text cannot be used directly by KNN. It must first be converted into numerical vectors.

The text features will be transformed with TF-IDF, while numeric columns will be scaled so that large values such as subscribers do not dominate the distance calculation.

In [7]:
tfidf = TfidfVectorizer(stop_words="english", max_features=1000)
text_matrix = tfidf.fit_transform(df[text_feature].fillna(""))

print(f"Text matrix shape: {text_matrix.shape[0]} courses x {text_matrix.shape[1]} terms")

Text matrix shape: 3672 courses x 1000 terms


In [8]:
scaler = StandardScaler()
numeric_matrix = scaler.fit_transform(df[numeric_features].fillna(0))
numeric_weight = 0.2
weighted_numeric_matrix = numeric_matrix * numeric_weight

print(f"Numeric matrix shape: {numeric_matrix.shape[0]} courses x {numeric_matrix.shape[1]} numeric features")
print(f"Numeric feature weight: {numeric_weight}")

Numeric matrix shape: 3672 courses x 4 numeric features
Numeric feature weight: 0.2


In [9]:
feature_matrix = np.hstack([text_matrix.toarray(), weighted_numeric_matrix])

print(f"Combined KNN feature matrix shape: {feature_matrix.shape[0]} courses x {feature_matrix.shape[1]} features")

Combined KNN feature matrix shape: 3672 courses x 1004 features


The final feature matrix has one row per course. The first part of the matrix represents course text, while the last columns represent scaled numeric attributes.

The numeric columns are down-weighted so that popularity and price support the recommendations without overpowering the course text.

**Scalability note:** This dense feature matrix is appropriate for the current dataset size. For much larger course catalogs, sparse matrix techniques could reduce memory usage during feature combination and nearest-neighbor search.

This keeps the KNN model content-based and avoids using synthetic users or collaborative filtering at this stage.

## 5. Build KNN Model

The KNN model will use scikit-learn to find the nearest courses to a selected course.

The final parameters will be documented after testing a simple working setup.

In [10]:
knn_model = NearestNeighbors(
    n_neighbors=6,
    metric="cosine",
    algorithm="brute",
)

knn_model.fit(feature_matrix)

print("KNN model fitted successfully.")

KNN model fitted successfully.


The model uses cosine distance because the feature matrix contains many text-based TF-IDF values. A brute-force search is acceptable for this dataset size and keeps the method easy to understand.

**Similarity score note:** The displayed KNN similarity score is derived from the cosine distance returned by the model, using `similarity = 1 - distance`.

`n_neighbors` is set to `6` so that the selected course itself can be removed and five recommendations can still be returned.

## 6. Recommendation Function

The recommendation function will accept a course title and return the top `n` most similar courses.

The function should hide the selected course itself from the output and return a readable table with course metadata and similarity information.

In [11]:
title_to_index = pd.Series(df.index, index=df["course_title"].str.lower())


def recommend_courses(course_title, n=5):
    """Return the top n courses most similar to the selected course title."""
    normalized_title = course_title.lower()

    if normalized_title not in title_to_index:
        raise ValueError(f"Course title not found: {course_title}")

    course_index = title_to_index[normalized_title]
    if isinstance(course_index, pd.Series):
        course_index = course_index.iloc[0]

    distances, indices = knn_model.kneighbors(feature_matrix[course_index].reshape(1, -1), n_neighbors=n + 1)

    recommendation_rows = []
    for distance, index in zip(distances[0], indices[0]):
        if index == course_index:
            continue

        recommendation_rows.append(
            {
                "course_title": df.loc[index, "course_title"],
                "subject": df.loc[index, "subject"],
                "level": df.loc[index, "level"],
                "price": df.loc[index, "price"],
                "num_subscribers": df.loc[index, "num_subscribers"],
                "distance": round(float(distance), 4),
                "similarity": round(1 - float(distance), 4),
            }
        )

        if len(recommendation_rows) == n:
            break

    return pd.DataFrame(recommendation_rows)

### Recommendation Function Edge Cases

If the selected course title is not found, the function raises an error. If duplicate titles exist, the first matching course is used, which keeps the behavior consistent and simple for this project stage.

## 7. Example Recommendations

The recommender will be tested with examples from different subject areas:

- Python-related course
- Business or finance course
- Web development course

Short observations will be added after the results.

In [12]:
python_course = "web programming with python"

print(f"Recommendations for: {python_course}")
recommend_courses(python_course, n=5)

Recommendations for: web programming with python


,course_title,subject,level,price,num_subscribers,distance,similarity
0,python web programming,web development,beginner level,100,1020,0.2541,0.7459
1,introduction to qgis python programming,web development,beginner level,85,197,0.3670,0.6330
2,learn html5 programming from scratch,web development,all levels,0,268923,0.4013,0.5987
3,python for beginners python programming langua...,web development,beginner level,150,6153,0.4117,0.5883
4,coding for entrepreneurs basic,web development,beginner level,0,161029,0.4318,0.5682


The Python example returns mostly web development and programming-related courses. The closest recommendation is another Python web programming course, which shows that the text component is influencing the KNN distance correctly.

Some broader web development courses also appear because they share subject and beginner-level signals with the selected course.

In [13]:
business_course = "ultimate investment banking course"

print(f"Recommendations for: {business_course}")
recommend_courses(business_course, n=5)

Recommendations for: ultimate investment banking course


,course_title,subject,level,price,num_subscribers,distance,similarity
0,the complete investment banking course 2017,business finance,all levels,195,8575,0.2448,0.7552
1,the investment banking recruitment series,business finance,all levels,40,17,0.4102,0.5898
2,advanced accounting for investment banking,business finance,intermediate level,50,1260,0.4430,0.5570
3,investment banking how to land a job on wall s...,business finance,all levels,75,1218,0.5303,0.4697
4,coaching course investment analysis for your c...,business finance,intermediate level,200,1,0.5418,0.4582


The business example returns courses from the Business Finance subject, especially investment banking and investment-related courses. This is a strong result for a content-based KNN model because the recommendations match both the topic and the subject area.

In [14]:
web_development_course = "learn complete web development from scratch"

print(f"Recommendations for: {web_development_course}")
recommend_courses(web_development_course, n=5)

Recommendations for: learn complete web development from scratch


,course_title,subject,level,price,num_subscribers,distance,similarity
0,backbone tutorial learn backbonejs from scratch,web development,all levels,90,4957,0.3761,0.6239
1,learn javascript from scratch become top rated...,web development,all levels,65,2570,0.3898,0.6102
2,learn wcf and web apis from scratch,web development,intermediate level,50,5398,0.3960,0.6040
3,learn javascript from scratch,web development,beginner level,20,4193,0.3962,0.6038
4,learn how to become a web developer from scratch,web development,beginner level,200,7062,0.4081,0.5919


The web development example returns courses about web development, JavaScript, HTML/CSS, and learning from scratch. The recommendations are not identical copies of the input course, but they are close enough to be useful alternatives for a learner interested in the same area.

## 8. Strengths and Limitations

### Strengths

- Simple and understandable: KNN recommends courses by finding nearby course vectors.
- Content-based: the model does not need user ratings or historical learner behavior.
- Flexible feature design: text and numeric course attributes can be combined in one feature matrix.
- Interpretable output: distance and similarity scores make it possible to inspect why courses are close.

### Limitations

- Recommendation quality depends strongly on feature engineering and feature weights.
- The model cannot learn personal preferences because no real user interaction data is used.
- Popularity, price, and duration can influence the distance even when the topic is not identical.
- KNN can become slower on much larger datasets because it compares the selected course with many other courses.

For this project stage, these limitations are acceptable because the task is to build a working content-based KNN pipeline before adding more advanced evaluation or user-profile logic.

## 9. Conclusions

This notebook created a working KNN-based content recommender for the Udemy course dataset.

The pipeline loads the processed course data, transforms course text with TF-IDF, scales numeric attributes, combines all features into one matrix, and uses `NearestNeighbors` to retrieve the most similar courses.

The example recommendations show that KNN can produce reasonable course suggestions across Python, Business Finance, and Web Development examples. The numeric feature weight helps keep the recommendations focused on course content while still allowing popularity, price, and duration to provide additional context.

This KNN notebook can later be compared with the TF-IDF recommender in `03_tfidf_recommender.ipynb` during evaluation.